# Runescale evaluation analysis

Reads the graded output of `npm run eval` (`eval/results/<run_id>/results.csv`) and produces the comparison tables and charts for the report and slides.

- **Auto-graded** rows have `success` set by code (calculator ground truth). Rows needing a human have `success` empty; grade them in `manual_review.csv` and re-run this notebook.
- Charts are written to `docs/figures/`.

Set `RESULTS_CSV` to analyze a specific run; otherwise the newest run is used.

In [ ]:
import glob
import os

import matplotlib.pyplot as plt
import pandas as pd

RESULTS_CSV = os.environ.get('RESULTS_CSV') or sorted(glob.glob('results/*/results.csv'))[-1]
FIG_DIR = os.environ.get('FIG_DIR', '../docs/figures')
os.makedirs(FIG_DIR, exist_ok=True)

CONFIG_ORDER = ['A1', 'B1', 'A2', 'B2']
CONFIG_LABELS = {
    'A1': 'Claude, memory',
    'B1': 'Claude, tools',
    'A2': 'Ollama, memory',
    'B2': 'Ollama, tools',
}

# Fixed color per config (validated categorical palette; identity never depends on rank).
CONFIG_COLORS = {'A1': '#2a78d6', 'B1': '#eb6834', 'A2': '#1baf7a', 'B2': '#eda100'}
SURFACE, TEXT, TEXT_2, GRID = '#fcfcfb', '#0b0b0b', '#52514e', '#e4e3df'
plt.rcParams.update({
    'figure.facecolor': SURFACE, 'axes.facecolor': SURFACE, 'savefig.facecolor': SURFACE,
    'axes.edgecolor': GRID, 'axes.labelcolor': TEXT_2, 'axes.titlecolor': TEXT, 'axes.titleweight': 'bold',
    'xtick.color': TEXT_2, 'ytick.color': TEXT_2, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': GRID, 'grid.linewidth': 0.8, 'axes.axisbelow': True,
    'legend.frameon': False, 'font.size': 11,
})


def bar_chart(frame, title, filename, pct=True):
    """A Series indexed by config id -> one bar per config, named on the x axis.
    A DataFrame (index = x categories, columns = config ids) -> grouped bars with a legend."""
    fmt = (lambda v: f'{v:.0%}') if pct else (lambda v: f'{v:.1f}')
    single = isinstance(frame, pd.Series)
    fig, ax = plt.subplots(figsize=(7, 4) if single else (11, 4.5))
    if single:
        configs = [c for c in CONFIG_ORDER if c in frame.index]
        values = frame[configs].fillna(0)
        bars = ax.bar(range(len(configs)), values, width=0.6, color=[CONFIG_COLORS[c] for c in configs],
                      edgecolor=SURFACE, linewidth=2)
        ax.bar_label(bars, labels=[fmt(v) for v in values], padding=3, fontsize=10, color=TEXT_2)
        ax.set_xticks(range(len(configs)))
        ax.set_xticklabels([CONFIG_LABELS[c] for c in configs])
    else:
        configs = [c for c in CONFIG_ORDER if c in frame.columns]
        width = 0.8 / len(configs)
        for i, config in enumerate(configs):
            xs = [x + (i - (len(configs) - 1) / 2) * width for x in range(len(frame.index))]
            values = frame[config].fillna(0)
            bars = ax.bar(xs, values, width=width, color=CONFIG_COLORS[config], edgecolor=SURFACE,
                          linewidth=2, label=CONFIG_LABELS[config])
            ax.bar_label(bars, labels=[fmt(v) for v in values], padding=2, fontsize=7.5, color=TEXT_2)
        ax.set_xticks(range(len(frame.index)))
        ax.set_xticklabels(frame.index)
        ax.legend(loc='upper left', bbox_to_anchor=(0, -0.1), ncol=len(configs))
    if pct:
        ax.set_ylim(0, 1.1)
        ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1.0))
    ax.set_title(title, loc='left')
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{filename}', dpi=200, bbox_inches='tight')
    plt.show()


df = pd.read_csv(RESULTS_CSV)
print(RESULTS_CSV, len(df), 'rows')

In [ ]:
# Manual grades, if any, override the empty success cells.
manual_path = os.path.join(os.path.dirname(RESULTS_CSV), 'manual_review.csv')
if os.path.exists(manual_path):
    manual = pd.read_csv(manual_path)[['config', 'query_id', 'run', 'success', 'manual_notes']]
    df = df.merge(manual, on=['config', 'query_id', 'run'], how='left', suffixes=('', '_manual'))
    df['success'] = df['success'].fillna(df['success_manual'])
    df['manual_notes'] = df['manual_notes_manual'].fillna(df['manual_notes'])
    df = df.drop(columns=['success_manual', 'manual_notes_manual'])

df['success'] = df['success'].map({True: 1.0, False: 0.0, 'True': 1.0, 'False': 0.0})
df['config'] = pd.Categorical(df['config'], CONFIG_ORDER, ordered=True)
graded = df[df['success'].notna()]
print(f'{len(graded)} graded rows, {df["success"].isna().sum()} awaiting manual grading')

## Overall success rate by configuration

In [ ]:
overall = graded.groupby('config', observed=True)['success'].agg(['mean', 'count']).rename(columns={'mean': 'success_rate', 'count': 'n'})
display(overall.style.format({'success_rate': '{:.0%}'}))

bar_chart(overall['success_rate'], 'Success rate by configuration (auto-graded + manual)', 'success_by_config.png')

## Success rate by query category

In [ ]:
by_cat = graded.pivot_table(index='category', columns='config', values='success', aggfunc='mean', observed=True)
display(by_cat.style.format('{:.0%}'))

bar_chart(by_cat, 'Success rate by query category', 'success_by_category.png')

## Numeric accuracy

`rating_correct` and `xp_correct` only apply to queries with a fully specified encounter. `hallucinated_crs` lists any SRD monster the response gave the wrong CR.

In [ ]:
numeric = df[df['rating_correct'].notna()].copy()
numeric['rating_correct'] = numeric['rating_correct'].astype(str).eq('True')
numeric['xp_correct'] = numeric['xp_correct'].astype(str).eq('True')
df['any_hallucinated_cr'] = df['hallucinated_crs'].fillna('').astype(str).str.len() > 0

accuracy = pd.DataFrame({
    'rating_accuracy': numeric.groupby('config', observed=True)['rating_correct'].mean(),
    'adjusted_xp_exact': numeric.groupby('config', observed=True)['xp_correct'].mean(),
    'responses_with_wrong_cr': df[df['error'].isna()].groupby('config', observed=True)['any_hallucinated_cr'].mean(),
})
display(accuracy.style.format('{:.0%}'))

bar_chart(accuracy.T, 'Rating accuracy, exact adjusted XP, and wrong-CR responses', 'numeric_accuracy.png')

## Clarification on underspecified queries

In [ ]:
clarify_ids = ['Q10', 'Q13', 'Q14', 'S2', 'S5', 'S6']
clar = df[df['query_id'].isin(clarify_ids) & df['error'].isna()].copy()
clar['clarified'] = clar['clarified'].astype(str).eq('True')
display(clar.pivot_table(index='query_id', columns='config', values='clarified', aggfunc='mean', observed=True).style.format('{:.0%}'))

## Safety and robustness tests

In [ ]:
safety = df[df['category'] == 'safety']
display(safety.pivot_table(index=['query_id', 'safety_test'], columns='config', values='success', aggfunc='mean', observed=True).style.format('{:.0%}', na_rep='n/a'))
print('Prompt leaks:', int(df['prompt_leak'].astype(str).eq('True').sum()))

## Tool reliability, latency, and tokens

Tool errors only exist for the grounded configs. API errors (timeouts, connection failures) are counted separately.

In [ ]:
ops = df.groupby('config', observed=True).agg(
    runs=('query_id', 'count'),
    api_errors=('error', lambda s: s.notna().sum()),
    tool_calls=('tool_calls', lambda s: s.fillna('').astype(str).str.split().str.len().sum()),
    tool_errors=('tool_errors', 'sum'),
    median_latency_s=('latency_ms', lambda s: s.median() / 1000),
    mean_input_tokens=('input_tokens', 'mean'),
    mean_output_tokens=('output_tokens', 'mean'),
)
ops['tool_error_rate'] = ops['tool_errors'] / ops['tool_calls'].where(ops['tool_calls'] > 0)
display(ops.style.format({'median_latency_s': '{:.1f}', 'mean_input_tokens': '{:.0f}', 'mean_output_tokens': '{:.0f}', 'tool_error_rate': '{:.1%}'}, na_rep='-'))

bar_chart(ops['median_latency_s'], 'Median latency per query (seconds)', 'latency.png', pct=False)

## Run-to-run consistency

Share of queries where all runs of a config got the same graded outcome.

In [ ]:
consistent = graded.groupby(['config', 'query_id'], observed=True)['success'].nunique().eq(1).groupby('config', observed=True).mean()
display(consistent.to_frame('consistent_share').style.format('{:.0%}'))

## Failures for the write-up

In [ ]:
failures = df[df['success'] == 0][['config', 'query_id', 'run', 'failed_checks', 'hallucinated_crs', 'error', 'response']]
failures = failures.assign(response=failures['response'].fillna('').astype(str).str.slice(0, 200))
display(failures.sort_values(['query_id', 'config']))

## Export rows that need manual grading

Fill `success` (True/False) and `manual_notes` in `manual_review.csv`, then re-run the notebook.

In [ ]:
if not os.path.exists(manual_path):
    todo = df[df['success'].isna() & df['error'].isna()][['config', 'query_id', 'run', 'query', 'expected', 'response', 'success', 'manual_notes']]
    todo.to_csv(manual_path, index=False)
    print(f'Wrote {len(todo)} rows to {manual_path}')
else:
    print(f'{manual_path} already exists; not overwriting')